In [1]:
conda_lib <- "/data/ag_reichart_lindberg/nhadizad/miniconda3/envs/old_R_433_for_cellchat_161/lib/R/library"
pkgs <- c("Seurat", "CellChat", "Matrix", "SeuratObject", "tidyr", "scales", "farver", "patchwork", "ggplot2")
for(p in pkgs) {
    if(file.exists(file.path(conda_lib, p))) {
        print(paste0(p, ": ", packageVersion(p, lib.loc = conda_lib)))
    } else {
        print(paste0(p, ": NOT FOUND"))
    }
}

[1] "Seurat: 5.2.1"
[1] "CellChat: 1.6.1"
[1] "Matrix: 1.6.5"
[1] "SeuratObject: 5.2.0"
[1] "tidyr: 1.3.2"
[1] "scales: 1.4.0"
[1] "farver: 2.1.2"
[1] "patchwork: 1.3.2"
[1] "ggplot2: 3.5.2"


In [2]:
# 1. Lock the library path to ensure it uses your working environment
conda_lib <- "/data/ag_reichart_lindberg/nhadizad/miniconda3/envs/old_R_433_for_cellchat_161/lib/R/library"
.libPaths(conda_lib)

# 2. Quiet down R's verbose warnings and messages
options(warn = -1, message = FALSE)

# 3. Load libraries cleanly
suppressPackageStartupMessages({
  library(Seurat)
  library(Matrix)
    library(CellChat)
    library(patchwork)
library(scales)
library(ggplot2)
})
set.seed(10)
options(future.globals.maxSize = 10 * 1024^3)
options(future.rng.onMisuse = "ignore")
plan("multicore", workers = 40)
setwd("/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_cellchat_obj/objects_cellchat_fractions/")


# Creating Cellchat fractions for each Condition

# 3month protocol

In [3]:
# Define global database once
CellChatDB <- CellChatDB.human

# --- 1. 3month-protocol ---
print("Processing: 3month-protocol")
input_3month <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_fraction_3month-protocol.rds"
seurat_3month <- readRDS(input_3month)

# Sanitize cell identities (replaces slashes and spaces with underscores to prevent CellChat errors)
seurat_3month$cell_type <- gsub("[/ ]", "_", seurat_3month$cell_type)

# Initialize and Process
cc_3month <- createCellChat(object = seurat_3month, group.by = "cell_type")
cc_3month@DB <- CellChatDB
cc_3month <- subsetData(cc_3month)

# Calculate Network
cc_3month <- identifyOverExpressedGenes(cc_3month)
cc_3month <- identifyOverExpressedInteractions(cc_3month)
cc_3month@idents <- droplevels(cc_3month@idents)
cc_3month <- computeCommunProb(cc_3month, type = "triMean")
cc_3month <- filterCommunication(cc_3month, min.cells = 5)
cc_3month <- computeCommunProbPathway(cc_3month)
cc_3month <- aggregateNet(cc_3month)
cc_3month <- netAnalysis_computeCentrality(cc_3month, slot.name = "netP")

# Save Output
saveRDS(cc_3month, file = "CellChat_obj_3month-protocol_netP.RDS")
print("Finished: 3month-protocol")

# Clear memory
rm(seurat_3month, cc_3month)
invisible(gc())


[1] "Processing: 3month-protocol"
[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  CM EC EC_papio FB Lymph_papio MC Mye_papio Mye_sus NC 
triMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2026-07-20 18:16:57.44194]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2026-07-20 18:36:56.307821]"
[1] "Finished: 3month-protocol"


# Rejection

In [4]:
# Define global database once
CellChatDB <- CellChatDB.human

# --- 2. Rejection ---
print("Processing: Rejection")
input_rejection <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_fraction_Rejection.rds"
seurat_rejection <- readRDS(input_rejection)

# Sanitize cell identities
seurat_rejection$cell_type <- gsub("[/ ]", "_", seurat_rejection$cell_type)

# Initialize and Process
cc_rejection <- createCellChat(object = seurat_rejection, group.by = "cell_type")
cc_rejection@DB <- CellChatDB
cc_rejection <- subsetData(cc_rejection)

# Calculate Network
cc_rejection <- identifyOverExpressedGenes(cc_rejection)
cc_rejection <- identifyOverExpressedInteractions(cc_rejection)
cc_rejection@idents <- droplevels(cc_rejection@idents)
cc_rejection <- computeCommunProb(cc_rejection, type = "triMean")
cc_rejection <- filterCommunication(cc_rejection, min.cells = 5)
cc_rejection <- computeCommunProbPathway(cc_rejection)
cc_rejection <- aggregateNet(cc_rejection)
cc_rejection <- netAnalysis_computeCentrality(cc_rejection, slot.name = "netP")

# Save Output
saveRDS(cc_rejection, file = "CellChat_obj_Rejection_netP.RDS")
print("Finished: Rejection")

# Clear memory
rm(seurat_rejection, cc_rejection)
invisible(gc())


[1] "Processing: Rejection"
[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  CM EC EC_papio FB Lymph_papio Lymph_sus MC Mye_papio Mye_sus NC 
triMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2026-07-22 10:27:56.052631]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2026-07-22 10:52:50.366176]"
The cell-cell communication related with the following cell groups are excluded due to the few number of cells:  Lymph_sus 
[1] "Finished: Rejection"


# 6month protocol

In [5]:
# Define global database once
CellChatDB <- CellChatDB.human

# --- 3. 6month-protocol ---
print("Processing: 6month-protocol")
input_6month <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_fraction_6month-protocol.rds"
seurat_6month <- readRDS(input_6month)

# Sanitize cell identities
seurat_6month$cell_type <- gsub("[/ ]", "_", seurat_6month$cell_type)

# Initialize and Process
cc_6month <- createCellChat(object = seurat_6month, group.by = "cell_type")
cc_6month@DB <- CellChatDB
cc_6month <- subsetData(cc_6month)

# Calculate Network
cc_6month <- identifyOverExpressedGenes(cc_6month)
cc_6month <- identifyOverExpressedInteractions(cc_6month)
cc_6month@idents <- droplevels(cc_6month@idents)
cc_6month <- computeCommunProb(cc_6month, type = "triMean")
cc_6month <- filterCommunication(cc_6month, min.cells = 5)
cc_6month <- computeCommunProbPathway(cc_6month)
cc_6month <- aggregateNet(cc_6month)
cc_6month <- netAnalysis_computeCentrality(cc_6month, slot.name = "netP")

# Save Output
saveRDS(cc_6month, file = "CellChat_obj_6month-protocol_netP.RDS")
print("Finished: 6month-protocol")

# Clear memory
rm(seurat_6month, cc_6month)
invisible(gc())


[1] "Processing: 6month-protocol"
[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  CM EC EC_papio FB Lymph_papio Lymph_sus MC Mye_papio Mye_sus NC 
triMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2026-07-22 10:54:42.297382]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2026-07-22 11:19:59.712004]"
The cell-cell communication related with the following cell groups are excluded due to the few number of cells:  Lymph_sus 
[1] "Finished: 6month-protocol"


# PCMV/PRV

In [6]:
# Define global database once
CellChatDB <- CellChatDB.human

# --- 4. PCMV/PRV ---
print("Processing: PCMV/PRV")
# Notice the filename is loaded as PCMV_PRV because of how we saved it earlier to avoid Linux path errors
input_pcmv <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_fraction_PCMV_PRV.rds"
seurat_pcmv <- readRDS(input_pcmv)

# Sanitize cell identities
seurat_pcmv$cell_type <- gsub("[/ ]", "_", seurat_pcmv$cell_type)

# Initialize and Process
cc_pcmv <- createCellChat(object = seurat_pcmv, group.by = "cell_type")
cc_pcmv@DB <- CellChatDB
cc_pcmv <- subsetData(cc_pcmv)

# Calculate Network
cc_pcmv <- identifyOverExpressedGenes(cc_pcmv)
cc_pcmv <- identifyOverExpressedInteractions(cc_pcmv)
cc_pcmv@idents <- droplevels(cc_pcmv@idents)
cc_pcmv <- computeCommunProb(cc_pcmv, type = "triMean")
cc_pcmv <- filterCommunication(cc_pcmv, min.cells = 5)
cc_pcmv <- computeCommunProbPathway(cc_pcmv)
cc_pcmv <- aggregateNet(cc_pcmv)
cc_pcmv <- netAnalysis_computeCentrality(cc_pcmv, slot.name = "netP")

# Save Output (Saving as PCMV_PRV for system safety)
saveRDS(cc_pcmv, file = "CellChat_obj_PCMV_PRV_netP.RDS")
print("Finished: PCMV/PRV")

# Clear memory
rm(seurat_pcmv, cc_pcmv)
invisible(gc())

[1] "Processing: PCMV/PRV"
[1] "Create a CellChat object from a Seurat object"
The `data` slot in the default assay is used. The default assay is RNA 
The `meta.data` slot in the Seurat object is used as cell meta information 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  CM EC FB Lymph_papio MC Mye_papio Mye_sus NC 
triMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sc/snRNA-seq data <<< [2026-07-22 11:21:21.432337]"
[1] ">>> CellChat inference is done. Parameter values are stored in `object@options$parameter` <<< [2026-07-22 11:42:52.037776]"
[1] "Finished: PCMV/PRV"
